# 全球半导体销售来源资格

固定抓取18页目录、最多160篇2017年后的公告，核对发布与修改时间。销售为三个月平均，不误称单月原始销量。不训练、不回测；先验证历史版本。

In [1]:
from pathlib import Path
import os, json, hashlib
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
import qlib, pandas as pd, yaml
from qlib.data import D
from qlib.workflow import R
from src.data.sia_sales_sources import collect_sia_sources
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace'); card=root/'research/sia_sales_source_20261013'
plan=json.loads((card/'plan.json').read_text())
raw=root/plan['raw_cache']
manifest=collect_sia_sources(raw,plan)
qlib.init(provider_uri=str(root/plan['price_provider']),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(freq='1min')).normalize().unique().sort_values()
entries=pd.DataFrame(manifest['entries'])
for key in ['published_at_utc','modified_at_utc']:
    entries[key]=pd.to_datetime(entries[key],utc=True)
entries['earliest_conservative_day']=entries.modified_at_utc.dt.tz_convert('America/New_York').dt.tz_localize(None).dt.normalize().map(lambda d: calendar[calendar>d][0] if (calendar>d).any() else pd.NaT)
audit={'candidate_articles':manifest['candidate_articles'],'parsed_articles':len(entries),'failures':manifest['failures'],
       'native_calendar_rows':len(calendar),'published_start':str(entries.published_at_utc.min()),'published_end':str(entries.published_at_utc.max()),
       'modification_clock_unknown':int(entries.modified_at_utc.isna().sum()),
       'modified_after_publication':int(entries.modification_lag_days.gt(0).sum()),
       'modified_more_than_30_days_later':int(entries.modification_lag_days.gt(30).sum()),
       'maximum_modification_lag_days':float(entries.modification_lag_days.max()),
       'monthly_average_explicit':int(entries.monthly_three_month_average_explicit.sum()),
       'original_historical_text_certified':False,'source_promotion_eligible':False,'model_fits':0,'backtests':0}
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],recorder_name='sia_publication_vintage_inventory',uri=str(tracking)):
    rec=R.get_recorder()
    rec.log_params(plan_sha256=hashlib.sha256((card/'plan.json').read_bytes()).hexdigest(),model_fits=0,backtests=0)
    rec.save_objects(artifact_path='sources',manifest=manifest,raw_documents={p.name:p.read_bytes() for p in raw.glob('*.html')},acquisition_code=(root/'src/data/sia_sales_sources.py').read_text())
    rec.save_objects(artifact_path='analysis',inventory=entries,native_calendar=calendar,audit=audit)
    rec.log_metrics(registered_artifact_count=6,source_articles=len(entries))
summary={'experiment':plan['experiment'],'decision':'needs_original_publication_vintages','objective_achieved':False,'goal_remains_active':True,
    'audit':audit,'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(tracking),
    'recorder_local_dir':str(tracking/str(rec.experiment_id)/rec.id),'provider_uri':str(root/plan['price_provider']),'region':'us','as_of':plan['as_of'],'qlib_docs':plan['qlib_docs'],
    'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['sources/*','analysis/inventory','analysis/native_calendar']}}
artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts'
verification={'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),'bytes':p.stat().st_size} for p in sorted(artifacts.rglob('*')) if p.is_file()]}
(card/'verification_manifest.json').write_text(json.dumps(verification,indent=2)+'\n')
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id))
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':plan['experiment'],'provider_uri':summary['provider_uri'],'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']}))
entries.drop(columns=['candidate_sales_paragraphs']).to_csv(card/'publication_inventory.csv',index=False)
audit


{'candidate_articles': 107,
 'parsed_articles': 107,
 'failures': [],
 'native_calendar_rows': 2008,
 'published_start': '2017-03-14 19:00:00+00:00',
 'published_end': '2026-10-05 12:00:17+00:00',
 'modification_clock_unknown': 56,
 'modified_after_publication': 51,
 'modified_more_than_30_days_later': 14,
 'maximum_modification_lag_days': 753.1989236111111,
 'monthly_average_explicit': 102,
 'original_historical_text_certified': False,
 'source_promotion_eligible': False,
 'model_fits': 0,
 'backtests': 0}

# 全球半导体销售来源资格结论

已执行18页目录抓取并冻结107篇候选公告、125份HTML原始文件；原始内容和时间元数据保存在NAS Qlib Recorder。解析107篇，下载/解析失败0篇。没有训练、预测或组合回测，稳定超额目标未达成。

公告覆盖2017-03-14 19:00:00+00:00至2026-10-05 12:00:17+00:00。51篇明确有后续修改，其中14篇相隔超过30天，最大753.20天；56篇缺少修改时间，保留未知，不推断“从未修改”。102篇正文明确注明月度销售为三个月移动平均。不能把观测月份当发布日，不能把今天的历史汇总当原始版本。

Qlib原生价格日历2008天用于时间映射。任何修改后版本只能考虑修改时点之后的交易日，且网站时间元数据不能单独证明当时文本内容。当前决定为`needs_original_publication_vintages`，来源未获历史原始版本资格，不能上线；不是模型无效的结论，也没有新收益结果。

初次资格运行把56篇缺少修改时间的公告列为解析失败；已修复为完整保留公告并将修改时间记为未知。初次Recorder及NAS归档在qualification_attempts.json保留，无模型重训或参数搜索。

下一步有具体可核查材料：105篇公告附有官方月度GSR图表PDF。核对附件自身年月、表格口径、生成时间及与公告金额的差异；历史网站迁移造成的路径日期不当作发布时间。若无法证实原始版本，只能保留条件性诊断或等待真实前向记录，不能给稳定超额背书。现有生产SIM未改动，目标保持active。
